In [1]:
import pandas as pd
pd.set_option('display.max_columns',None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [2]:
# Load the dataset
df = pd.read_csv('/Users/rghdmajd/PycharmProjects/ClinVar_Project/data/raw/clinvar_conflicting.csv',low_memory=False)

# Display the first 5 rows to understand the structure
df.head()

,CHROM,POS,REF,ALT,AF_ESP,AF_EXAC,AF_TGP,CLNDISDB,CLNDISDBINCL,CLNDN,CLNDNINCL,CLNHGVS,CLNSIGINCL,CLNVC,CLNVI,MC,ORIGIN,SSR,CLASS,Allele,Consequence,IMPACT,SYMBOL,Feature_type,Feature,BIOTYPE,EXON,INTRON,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,DISTANCE,STRAND,BAM_EDIT,SIFT,PolyPhen,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,LoFtool,CADD_PHRED,CADD_RAW,BLOSUM62
0,1,1168180,G,C,0.0771,0.10020,0.1066,MedGen:CN169374,NaN,not_specified,NaN,NC_000001.10:g.1168180G>C,NaN,single_nucleotide_variant,UniProtKB_(protein):Q96L58#VAR_059317,SO:0001583|missense_variant,1,NaN,0,C,missense_variant,MODERATE,B3GALT6,Transcript,NM_080605.3,protein_coding,1/1,NaN,552,522,174,E/D,gaG/gaC,NaN,1.0,NaN,tolerated,benign,NaN,NaN,NaN,NaN,NaN,1.053,-0.208682,2.0
1,1,1470752,G,A,0.0000,0.00000,0.0000,"MedGen:C1843891,OMIM:607454,Orphanet:ORPHA9877...",NaN,Spinocerebellar_ataxia_21|not_provided,NaN,NC_000001.10:g.1470752G>A,NaN,single_nucleotide_variant,OMIM_Allelic_Variant:616101.0001|UniProtKB_(pr...,SO:0001583|missense_variant,1,NaN,0,A,missense_variant,MODERATE,TMEM240,Transcript,NM_001114748.1,protein_coding,4/4,NaN,523,509,170,P/L,cCg/cTg,NaN,-1.0,OK,deleterious_low_confidence,benign,NaN,NaN,NaN,NaN,NaN,31.000,6.517838,-3.0
2,1,1737942,A,G,0.0000,0.00001,0.0000,"Human_Phenotype_Ontology:HP:0000486,MedGen:C00...",NaN,Strabismus|Nystagmus|Hypothyroidism|Intellectu...,NaN,NC_000001.10:g.1737942A>G,NaN,single_nucleotide_variant,OMIM_Allelic_Variant:139380.0002|UniProtKB_(pr...,"SO:0001583|missense_variant,SO:0001623|5_prime...",35,NaN,1,G,missense_variant,MODERATE,GNB1,Transcript,NM_002074.4,protein_coding,6/12,NaN,632,239,80,I/T,aTc/aCc,NaN,-1.0,OK,deleterious,probably_damaging,NaN,NaN,NaN,NaN,NaN,28.100,6.061752,-1.0
3,1,2160305,G,A,0.0000,0.00000,0.0000,"MedGen:C1321551,OMIM:182212,SNOMED_CT:83092002...",NaN,Shprintzen-Goldberg_syndrome|not_provided,NaN,NC_000001.10:g.2160305G>A,NaN,single_nucleotide_variant,OMIM_Allelic_Variant:164780.0004|UniProtKB_(pr...,SO:0001583|missense_variant,33,NaN,0,A,missense_variant,MODERATE,SKI,Transcript,XM_005244775.1,protein_coding,1/7,NaN,132,100,34,G/S,Ggc/Agc,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.500,3.114491,NaN
4,1,2160305,G,T,0.0000,0.00000,0.0000,"MedGen:C1321551,OMIM:182212,SNOMED_CT:83092002",NaN,Shprintzen-Goldberg_syndrome,NaN,NC_000001.10:g.2160305G>T,NaN,single_nucleotide_variant,OMIM_Allelic_Variant:164780.0005|UniProtKB_(pr...,SO:0001583|missense_variant,33,NaN,0,T,missense_variant,MODERATE,SKI,Transcript,XM_005244775.1,protein_coding,1/7,NaN,132,100,34,G/C,Ggc/Tgc,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.700,4.766224,-3.0


## 1. Handling Missing Values

### Automated Cleaning Process
I implemented a dynamic function to handle missing data efficiently:
- **Dropping:** Columns with > 20% missing values are removed to reduce noise.
- **Imputation:** Remaining missing values are filled using the **Median** for numerical features and the **Mode** for categorical features.

In [3]:
def clean_missing_values(df, threshold=0.2):
    """
    Cleans the dataframe by handling missing values based on a percentage threshold.
    - Drops columns if missing data > threshold.
    - Special handling for AF columns (fills with 0).
    - Imputes other numerical data with Median and categorical with Mode.
    """
    df_cleaned = df.copy()
    af_cols = ['AF_ESP', 'AF_EXAC', 'AF_TGP']
    
    missing_percentages = df_cleaned.isnull().mean()
    
    for col in df_cleaned.columns:
        # 1. Drop column if missing data exceeds the threshold
        if missing_percentages[col] > threshold:
            df_cleaned.drop(columns=[col], inplace=True)
            print(f"Dropped column: {col} (Missing: {missing_percentages[col]:.2%})")
            
        # 2. Impute missing data if it is present but below the threshold
        elif missing_percentages[col] > 0:
            
            # SPECIAL CASE: Allele Frequency columns (fill with 0)
            if col in af_cols:
                df_cleaned[col] = df_cleaned[col].fillna(0)
                print(f"Filled column: {col} with 0 (Special Biological Rule)")
                
            # NUMERICAL CASE: Fill with Median
            elif df_cleaned[col].dtype in ['float64', 'int64']:
                median_val = df_cleaned[col].median()
                df_cleaned[col] = df_cleaned[col].fillna(median_val)
                print(f"Filled column: {col} with Median")
                
            # CATEGORICAL CASE: Fill with Mode
            else:
                mode_val = df_cleaned[col].mode()[0]
                df_cleaned[col] = df_cleaned[col].fillna(mode_val)
                print(f"Filled column: {col} with Mode")
                
    return df_cleaned

# Apply the refined cleaning process
df = clean_missing_values(df)

# Display result info
print("\nCleaning complete. New dataframe shape:", df.shape)

Dropped column: CLNDISDBINCL (Missing: 99.74%)
Dropped column: CLNDNINCL (Missing: 99.74%)
Dropped column: CLNSIGINCL (Missing: 99.74%)
Dropped column: CLNVI (Missing: 57.57%)
Filled column: MC with Mode
Dropped column: SSR (Missing: 99.80%)
Filled column: SYMBOL with Mode
Filled column: Feature_type with Mode
Filled column: Feature with Mode
Filled column: BIOTYPE with Mode
Filled column: EXON with Mode
Dropped column: INTRON (Missing: 86.50%)
Filled column: cDNA_position with Mode
Filled column: CDS_position with Mode
Filled column: Protein_position with Mode
Filled column: Amino_acids with Mode
Filled column: Codons with Mode
Dropped column: DISTANCE (Missing: 99.83%)
Filled column: STRAND with Median
Dropped column: BAM_EDIT (Missing: 50.96%)
Dropped column: SIFT (Missing: 61.90%)
Dropped column: PolyPhen (Missing: 61.96%)
Dropped column: MOTIF_NAME (Missing: 100.00%)
Dropped column: MOTIF_POS (Missing: 100.00%)
Dropped column: HIGH_INF_POS (Missing: 100.00%)
Dropped column: MOTIF_

In [4]:
df.to_csv('/Users/rghdmajd/PycharmProjects/ClinVar_Project/data/processed/cleaned_semi_processedv1.csv', index=False) 

In [5]:
df.columns

Index(['CHROM', 'POS', 'REF', 'ALT', 'AF_ESP', 'AF_EXAC', 'AF_TGP', 'CLNDISDB',
       'CLNDN', 'CLNHGVS', 'CLNVC', 'MC', 'ORIGIN', 'CLASS', 'Allele',
       'Consequence', 'IMPACT', 'SYMBOL', 'Feature_type', 'Feature', 'BIOTYPE',
       'EXON', 'cDNA_position', 'CDS_position', 'Protein_position',
       'Amino_acids', 'Codons', 'STRAND', 'LoFtool', 'CADD_PHRED', 'CADD_RAW'],
      dtype='str')

In [6]:
df.head()

,CHROM,POS,REF,ALT,AF_ESP,AF_EXAC,AF_TGP,CLNDISDB,CLNDN,CLNHGVS,CLNVC,MC,ORIGIN,CLASS,Allele,Consequence,IMPACT,SYMBOL,Feature_type,Feature,BIOTYPE,EXON,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,STRAND,LoFtool,CADD_PHRED,CADD_RAW
0,1,1168180,G,C,0.0771,0.10020,0.1066,MedGen:CN169374,not_specified,NC_000001.10:g.1168180G>C,single_nucleotide_variant,SO:0001583|missense_variant,1,0,C,missense_variant,MODERATE,B3GALT6,Transcript,NM_080605.3,protein_coding,1/1,552,522,174,E/D,gaG/gaC,1.0,0.157,1.053,-0.208682
1,1,1470752,G,A,0.0000,0.00000,0.0000,"MedGen:C1843891,OMIM:607454,Orphanet:ORPHA9877...",Spinocerebellar_ataxia_21|not_provided,NC_000001.10:g.1470752G>A,single_nucleotide_variant,SO:0001583|missense_variant,1,0,A,missense_variant,MODERATE,TMEM240,Transcript,NM_001114748.1,protein_coding,4/4,523,509,170,P/L,cCg/cTg,-1.0,0.157,31.000,6.517838
2,1,1737942,A,G,0.0000,0.00001,0.0000,"Human_Phenotype_Ontology:HP:0000486,MedGen:C00...",Strabismus|Nystagmus|Hypothyroidism|Intellectu...,NC_000001.10:g.1737942A>G,single_nucleotide_variant,"SO:0001583|missense_variant,SO:0001623|5_prime...",35,1,G,missense_variant,MODERATE,GNB1,Transcript,NM_002074.4,protein_coding,6/12,632,239,80,I/T,aTc/aCc,-1.0,0.157,28.100,6.061752
3,1,2160305,G,A,0.0000,0.00000,0.0000,"MedGen:C1321551,OMIM:182212,SNOMED_CT:83092002...",Shprintzen-Goldberg_syndrome|not_provided,NC_000001.10:g.2160305G>A,single_nucleotide_variant,SO:0001583|missense_variant,33,0,A,missense_variant,MODERATE,SKI,Transcript,XM_005244775.1,protein_coding,1/7,132,100,34,G/S,Ggc/Agc,1.0,0.157,22.500,3.114491
4,1,2160305,G,T,0.0000,0.00000,0.0000,"MedGen:C1321551,OMIM:182212,SNOMED_CT:83092002",Shprintzen-Goldberg_syndrome,NC_000001.10:g.2160305G>T,single_nucleotide_variant,SO:0001583|missense_variant,33,0,T,missense_variant,MODERATE,SKI,Transcript,XM_005244775.1,protein_coding,1/7,132,100,34,G/C,Ggc/Tgc,1.0,0.157,24.700,4.766224


## 2. Dropping Irrelevant Features

### Constant columns

In [7]:
# 1. Identify constant columns (columns with only one unique value)
constant_cols = [col for col in df.columns if df[col].nunique() == 1]

print(f"Constant columns found: {constant_cols}")

# 2. Drop them if there are any
if constant_cols:
    df.drop(columns=constant_cols, inplace=True)
    print(f"Dropped {len(constant_cols)} constant columns.")
else:
    print("No constant columns to drop.")

# Check the new shape
print(f"Current shape: {df.shape}")

Constant columns found: []
No constant columns to drop.
Current shape: (65188, 31)


In [8]:
df.columns

Index(['CHROM', 'POS', 'REF', 'ALT', 'AF_ESP', 'AF_EXAC', 'AF_TGP', 'CLNDISDB',
       'CLNDN', 'CLNHGVS', 'CLNVC', 'MC', 'ORIGIN', 'CLASS', 'Allele',
       'Consequence', 'IMPACT', 'SYMBOL', 'Feature_type', 'Feature', 'BIOTYPE',
       'EXON', 'cDNA_position', 'CDS_position', 'Protein_position',
       'Amino_acids', 'Codons', 'STRAND', 'LoFtool', 'CADD_PHRED', 'CADD_RAW'],
      dtype='str')

In [9]:
df.head()

,CHROM,POS,REF,ALT,AF_ESP,AF_EXAC,AF_TGP,CLNDISDB,CLNDN,CLNHGVS,CLNVC,MC,ORIGIN,CLASS,Allele,Consequence,IMPACT,SYMBOL,Feature_type,Feature,BIOTYPE,EXON,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,STRAND,LoFtool,CADD_PHRED,CADD_RAW
0,1,1168180,G,C,0.0771,0.10020,0.1066,MedGen:CN169374,not_specified,NC_000001.10:g.1168180G>C,single_nucleotide_variant,SO:0001583|missense_variant,1,0,C,missense_variant,MODERATE,B3GALT6,Transcript,NM_080605.3,protein_coding,1/1,552,522,174,E/D,gaG/gaC,1.0,0.157,1.053,-0.208682
1,1,1470752,G,A,0.0000,0.00000,0.0000,"MedGen:C1843891,OMIM:607454,Orphanet:ORPHA9877...",Spinocerebellar_ataxia_21|not_provided,NC_000001.10:g.1470752G>A,single_nucleotide_variant,SO:0001583|missense_variant,1,0,A,missense_variant,MODERATE,TMEM240,Transcript,NM_001114748.1,protein_coding,4/4,523,509,170,P/L,cCg/cTg,-1.0,0.157,31.000,6.517838
2,1,1737942,A,G,0.0000,0.00001,0.0000,"Human_Phenotype_Ontology:HP:0000486,MedGen:C00...",Strabismus|Nystagmus|Hypothyroidism|Intellectu...,NC_000001.10:g.1737942A>G,single_nucleotide_variant,"SO:0001583|missense_variant,SO:0001623|5_prime...",35,1,G,missense_variant,MODERATE,GNB1,Transcript,NM_002074.4,protein_coding,6/12,632,239,80,I/T,aTc/aCc,-1.0,0.157,28.100,6.061752
3,1,2160305,G,A,0.0000,0.00000,0.0000,"MedGen:C1321551,OMIM:182212,SNOMED_CT:83092002...",Shprintzen-Goldberg_syndrome|not_provided,NC_000001.10:g.2160305G>A,single_nucleotide_variant,SO:0001583|missense_variant,33,0,A,missense_variant,MODERATE,SKI,Transcript,XM_005244775.1,protein_coding,1/7,132,100,34,G/S,Ggc/Agc,1.0,0.157,22.500,3.114491
4,1,2160305,G,T,0.0000,0.00000,0.0000,"MedGen:C1321551,OMIM:182212,SNOMED_CT:83092002",Shprintzen-Goldberg_syndrome,NC_000001.10:g.2160305G>T,single_nucleotide_variant,SO:0001583|missense_variant,33,0,T,missense_variant,MODERATE,SKI,Transcript,XM_005244775.1,protein_coding,1/7,132,100,34,G/C,Ggc/Tgc,1.0,0.157,24.700,4.766224


### Encoding and Dropping `MC`

After conducting our EDA, we performed this transformation for the following reasons:

- **Decoupling multi-label data:** the count plots revealed that `MC` contained multiple overlapping categories per cell, preventing the model from learning the distinct impact of individual variants.
- **Enhancing predictive power:** converting these into binary flags (`is_variant`) lets the model evaluate the specific influence of each variant type, which our bivariate analysis identified as a primary driver for classification.
- **Reducing noise & complexity:** dropping the original column eliminates noise from rare, uninformative combinations, mitigating overfitting and high dimensionality.
- **Model compatibility:** converting text-based descriptors into structured, numeric binary data is essential for training, since most algorithms cannot interpret raw, comma-separated strings.

In [10]:
# 1. Ensure the column contains strings to avoid errors during processing
df['MC'] = df['MC'].astype(str)

# 2. Extract all unique variant types to create dynamic binary flags
# Split by comma, explode the lists, and get unique values
all_variants = df['MC'].str.split(',').explode().unique()

print(f"Found {len(all_variants)} unique variant types: {all_variants}")

# 3. Create binary flag columns (One-Hot Encoding style for multi-label)
for variant in all_variants:
    # Clean the variant name to create a valid column header (replace special chars with underscores)
    col_name = f"is_{variant.replace(':', '_').replace(' ', '_')}"
    
    # Create the column: 1 if the variant is present in the string, 0 otherwise
    # We use .str.contains to handle the presence of the substring
    df[col_name] = df['MC'].str.contains(variant, regex=False).astype(int)

# 4. Drop the original multi-value column as the information is now encoded in new columns
df.drop(columns=['MC'], inplace=True)

# 5. Verification: Check the newly created columns
new_cols = [col for col in df.columns if col.startswith('is_')]
print(f"\nSuccessfully created {len(new_cols)} binary columns.")
print(f"Current dataframe shape: {df.shape}")

Found 11 unique variant types: <StringArray>
[       'SO:0001583|missense_variant',     'SO:0001623|5_prime_UTR_variant',
          'SO:0001627|intron_variant',     'SO:0001624|3_prime_UTR_variant',
    'SO:0001636|2KB_upstream_variant',      'SO:0001819|synonymous_variant',
 'SO:0001634|500B_downstream_variant',    'SO:0001575|splice_donor_variant',
      'SO:0001589|frameshift_variant',                'SO:0001587|nonsense',
 'SO:0001574|splice_acceptor_variant']
Length: 11, dtype: str

Successfully created 11 binary columns.
Current dataframe shape: (65188, 41)


### Encoding and Dropping `CLNDISDB` and `CLNDN`

We transformed both columns into numerical counts for the following reasons:

- **Converting unstructured data to quantifiable metrics:** both columns contained messy, multi-valued strings that lacked predictive utility in their raw, text-based format.
- **Feature engineering:** converting these strings into simple counts lets the model measure the relationship between variant complexity/disease burden and interpretation conflict as a clear, continuous numerical feature.
- **Mitigating high dimensionality:** dropping the original columns removes high-cardinality, unstructured text that would otherwise introduce excessive noise and memory-heavy dimensionality during training.
- **Optimizing model readiness:** numerical counts are directly compatible with machine learning algorithms, letting the model learn efficiently without complex NLP or text-parsing logic.

In [11]:
# Define a general function to count items in messy strings
def count_items(value, excluded=['not_specified', 'not_provided']):
    if pd.isna(value) or str(value).lower() in excluded:
        return 0
    # Split by common delimiters and filter excluded values
    items = str(value).replace(',', '|').split('|')
    clean_list = [i for i in items if i.strip().lower() not in excluded and i.strip() != '']
    return len(clean_list)

# Apply to both columns
df['Disease_Count'] = df['CLNDISDB'].apply(count_items)
df['Clinical_Disease_Count'] = df['CLNDN'].apply(count_items)

# Drop the original columns
df.drop(columns=['CLNDISDB', 'CLNDN'], inplace=True)

print("Engineered numerical features and dropped original unstructured columns.")

Engineered numerical features and dropped original unstructured columns.


### Dropping the `EXON` Column

We dropped `EXON` because it satisfies the criteria for removal in a high-quality predictive model:

- **Redundancy:** the genomic information it contains is often represented by more specific, actionable features such as `Consequence` or `IMPACT`, making the column repetitive.
- **Data noise:** high levels of missing or inconsistent values introduce noise that can mislead the model, reducing its ability to identify meaningful patterns.
- **Low predictive power:** this feature lacks sufficient variance or distinct informational value to improve the model's accuracy, making it an unnecessary dimension.
- **Simplifying model architecture:** removing this redundant column reduces the dataset's complexity and dimensionality, helping prevent overfitting and speeding up training.

In [12]:
if 'EXON' in df.columns:
    df.drop(columns=['EXON'], inplace=True)
    print("Column 'exon' dropped because it is redundant and noisy.")

Column 'exon' dropped because it is redundant and noisy.


In [13]:
df.head()

,CHROM,POS,REF,ALT,AF_ESP,AF_EXAC,AF_TGP,CLNHGVS,CLNVC,ORIGIN,CLASS,Allele,Consequence,IMPACT,SYMBOL,Feature_type,Feature,BIOTYPE,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,STRAND,LoFtool,CADD_PHRED,CADD_RAW,is_SO_0001583|missense_variant,is_SO_0001623|5_prime_UTR_variant,is_SO_0001627|intron_variant,is_SO_0001624|3_prime_UTR_variant,is_SO_0001636|2KB_upstream_variant,is_SO_0001819|synonymous_variant,is_SO_0001634|500B_downstream_variant,is_SO_0001575|splice_donor_variant,is_SO_0001589|frameshift_variant,is_SO_0001587|nonsense,is_SO_0001574|splice_acceptor_variant,Disease_Count,Clinical_Disease_Count
0,1,1168180,G,C,0.0771,0.10020,0.1066,NC_000001.10:g.1168180G>C,single_nucleotide_variant,1,0,C,missense_variant,MODERATE,B3GALT6,Transcript,NM_080605.3,protein_coding,552,522,174,E/D,gaG/gaC,1.0,0.157,1.053,-0.208682,1,0,0,0,0,0,0,0,0,0,0,1,0
1,1,1470752,G,A,0.0000,0.00000,0.0000,NC_000001.10:g.1470752G>A,single_nucleotide_variant,1,0,A,missense_variant,MODERATE,TMEM240,Transcript,NM_001114748.1,protein_coding,523,509,170,P/L,cCg/cTg,-1.0,0.157,31.000,6.517838,1,0,0,0,0,0,0,0,0,0,0,4,1
2,1,1737942,A,G,0.0000,0.00001,0.0000,NC_000001.10:g.1737942A>G,single_nucleotide_variant,35,1,G,missense_variant,MODERATE,GNB1,Transcript,NM_002074.4,protein_coding,632,239,80,I/T,aTc/aCc,-1.0,0.157,28.100,6.061752,1,1,0,0,0,0,0,0,0,0,0,48,23
3,1,2160305,G,A,0.0000,0.00000,0.0000,NC_000001.10:g.2160305G>A,single_nucleotide_variant,33,0,A,missense_variant,MODERATE,SKI,Transcript,XM_005244775.1,protein_coding,132,100,34,G/S,Ggc/Agc,1.0,0.157,22.500,3.114491,1,0,0,0,0,0,0,0,0,0,0,4,1
4,1,2160305,G,T,0.0000,0.00000,0.0000,NC_000001.10:g.2160305G>T,single_nucleotide_variant,33,0,T,missense_variant,MODERATE,SKI,Transcript,XM_005244775.1,protein_coding,132,100,34,G/C,Ggc/Tgc,1.0,0.157,24.700,4.766224,1,0,0,0,0,0,0,0,0,0,0,3,1


### Dropping the `CLNHGVS` Column

We dropped `CLNHGVS` to protect the integrity of the predictive model:

- **Preventing data leakage:** as a unique identifier for each variant, this column could let the model "memorize" specific training samples instead of learning generalized genomic patterns.
- **Mitigating overfitting:** retaining high-cardinality unique IDs can cause artificially high performance on training data while failing to generalize to unseen variants.
- **Eliminating non-predictive features:** unique identifiers provide no biological or clinical insight relevant to predicting interpretation conflicts.
- **Improving computational efficiency:** removing columns with no discriminatory power simplifies the model and reduces the risk of incorporating irrelevant noise.

In [14]:
# Drop the CLNHGVS column as it is a unique identifier causing data leakage
if 'CLNHGVS' in df.columns:
    df.drop(columns=['CLNHGVS'], inplace=True)
    print("Column 'CLNHGVS' dropped successfully.")

Column 'CLNHGVS' dropped successfully.


In [15]:
df.to_csv('/Users/rghdmajd/PycharmProjects/ClinVar_Project/data/processed/cleaned_semi_processedv2.csv', index=False)

### After heatmap analysis

## 3. Feature Engineering and Selection (Post-Heatmap)

- **Dimensionality reduction:** dropping redundant positional features (`cDNA_position`, `CDS_position`, `Protein_position`) and low-predictive columns (`CADD_RAW`) minimizes noise and mitigates overfitting.
- **Feature consolidation:** aggregating the separate allele-frequency columns into a single `mean_AF` feature reduces multicollinearity and gives a more robust, generalized representation of the variant's frequency across populations.
- **Model optimization:** eliminating high-correlation variables and consolidating sparse data lets the model focus on the most impactful features, leading to faster training and better predictive performance.

In [16]:
df.columns

Index(['CHROM', 'POS', 'REF', 'ALT', 'AF_ESP', 'AF_EXAC', 'AF_TGP', 'CLNVC',
       'ORIGIN', 'CLASS', 'Allele', 'Consequence', 'IMPACT', 'SYMBOL',
       'Feature_type', 'Feature', 'BIOTYPE', 'cDNA_position', 'CDS_position',
       'Protein_position', 'Amino_acids', 'Codons', 'STRAND', 'LoFtool',
       'CADD_PHRED', 'CADD_RAW', 'is_SO_0001583|missense_variant',
       'is_SO_0001623|5_prime_UTR_variant', 'is_SO_0001627|intron_variant',
       'is_SO_0001624|3_prime_UTR_variant',
       'is_SO_0001636|2KB_upstream_variant',
       'is_SO_0001819|synonymous_variant',
       'is_SO_0001634|500B_downstream_variant',
       'is_SO_0001575|splice_donor_variant',
       'is_SO_0001589|frameshift_variant', 'is_SO_0001587|nonsense',
       'is_SO_0001574|splice_acceptor_variant', 'Disease_Count',
       'Clinical_Disease_Count'],
      dtype='str')

In [17]:
cols_to_drop = [
    'cDNA_position', 
    'CDS_position', 
    'Protein_position', 
    'Disease_Count',
     'CADD_RAW' 
]

existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]


df = df.drop(columns=existing_cols_to_drop)


print(f" Dropped: {existing_cols_to_drop}")
print(f" Remain: {df.shape[1]}")

 Dropped: ['cDNA_position', 'CDS_position', 'Protein_position', 'Disease_Count', 'CADD_RAW']
 Remain: 34


In [18]:
af_cols = ['AF_ESP', 'AF_EXAC', 'AF_TGP']

existing_af = [col for col in af_cols if col in df.columns]

if len(existing_af) > 0:
    df['mean_AF'] = df[existing_af].mean(axis=1)
    
    df = df.drop(columns=existing_af)

In [19]:
df.columns

Index(['CHROM', 'POS', 'REF', 'ALT', 'CLNVC', 'ORIGIN', 'CLASS', 'Allele',
       'Consequence', 'IMPACT', 'SYMBOL', 'Feature_type', 'Feature', 'BIOTYPE',
       'Amino_acids', 'Codons', 'STRAND', 'LoFtool', 'CADD_PHRED',
       'is_SO_0001583|missense_variant', 'is_SO_0001623|5_prime_UTR_variant',
       'is_SO_0001627|intron_variant', 'is_SO_0001624|3_prime_UTR_variant',
       'is_SO_0001636|2KB_upstream_variant',
       'is_SO_0001819|synonymous_variant',
       'is_SO_0001634|500B_downstream_variant',
       'is_SO_0001575|splice_donor_variant',
       'is_SO_0001589|frameshift_variant', 'is_SO_0001587|nonsense',
       'is_SO_0001574|splice_acceptor_variant', 'Clinical_Disease_Count',
       'mean_AF'],
      dtype='str')

### Dropping `origin` and `biotype`

We dropped these columns to streamline the dataset:

- **Low informational value:** these features show low variance across the dataset, providing insufficient discriminatory power between classes.
- **Redundancy:** the information they provide is either captured by other, more predictive variables or is biologically irrelevant to identifying conflicting clinical interpretations.
- **Preventing noise:** removing these columns reduces noise in the input data, helping the model focus on high-impact features and reducing overfitting risk.
- **Improving model efficiency:** reducing dimensionality improves computational efficiency and lets the model learn patterns more effectively.

In [20]:
cols_to_drop = [
'ORIGIN' , 'BIOTYPE'
]

existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]


df = df.drop(columns=existing_cols_to_drop)


print(f" Dropped: {existing_cols_to_drop}")
print(f" Remain: {df.shape[1]}")

 Dropped: ['ORIGIN', 'BIOTYPE']
 Remain: 30


In [21]:
df.to_csv('/Users/rghdmajd/PycharmProjects/ClinVar_Project/data/processed/cleaned_semi_processedv3.csv', index=False)

In [22]:
df.columns

Index(['CHROM', 'POS', 'REF', 'ALT', 'CLNVC', 'CLASS', 'Allele', 'Consequence',
       'IMPACT', 'SYMBOL', 'Feature_type', 'Feature', 'Amino_acids', 'Codons',
       'STRAND', 'LoFtool', 'CADD_PHRED', 'is_SO_0001583|missense_variant',
       'is_SO_0001623|5_prime_UTR_variant', 'is_SO_0001627|intron_variant',
       'is_SO_0001624|3_prime_UTR_variant',
       'is_SO_0001636|2KB_upstream_variant',
       'is_SO_0001819|synonymous_variant',
       'is_SO_0001634|500B_downstream_variant',
       'is_SO_0001575|splice_donor_variant',
       'is_SO_0001589|frameshift_variant', 'is_SO_0001587|nonsense',
       'is_SO_0001574|splice_acceptor_variant', 'Clinical_Disease_Count',
       'mean_AF'],
      dtype='str')

## 4. Columns Renaming

In [23]:
# Create a mapping dictionary for renaming
new_names = {col: col.split('|')[-1] for col in df.columns if 'is_SO_' in col}

# Applying the changes
df.rename(columns=new_names, inplace=True)

# Optional: Further cleaning to make them standard lowercase/underscored
df.rename(columns=lambda x: x.lower().replace(' ', '_'), inplace=True)

print("Columns renamed successfully!")

Columns renamed successfully!


In [24]:
df.to_csv('/Users/rghdmajd/PycharmProjects/ClinVar_Project/data/processed/cleaned_semi_processedv4.csv', index=False) 

In [25]:
len(df.columns),df.columns

(30,
 Index(['chrom', 'pos', 'ref', 'alt', 'clnvc', 'class', 'allele', 'consequence',
        'impact', 'symbol', 'feature_type', 'feature', 'amino_acids', 'codons',
        'strand', 'loftool', 'cadd_phred', 'missense_variant',
        '5_prime_utr_variant', 'intron_variant', '3_prime_utr_variant',
        '2kb_upstream_variant', 'synonymous_variant', '500b_downstream_variant',
        'splice_donor_variant', 'frameshift_variant', 'nonsense',
        'splice_acceptor_variant', 'clinical_disease_count', 'mean_af'],
       dtype='str'))

In [26]:
# 1. Spatial & Genomic Location Features
location_cols = [
    'chrom', 'pos', 'ref', 'alt', 'amino_acids', 'codons', 'strand'
]

# 2. Population Frequency Features
frequency_cols = ['mean_af']

# 3. Pathogenicity & Impact Scores
score_cols = ['cadd_phred', 'loftool', 'impact']

# 4. Clinical & Contextual Features
clinical_cols = ['clnvc', 'symbol', 'clinical_disease_count']

# 5. Variant Consequence/Type Features
consequence_cols = [
    'consequence', 
    'feature_type', 
    'feature',  
    'allele',
    'missense_variant', 
    '5_prime_utr_variant',
    'intron_variant', 
    '3_prime_utr_variant',
    '2kb_upstream_variant', 
    'synonymous_variant',
    '500b_downstream_variant', 
    'splice_donor_variant',
    'frameshift_variant', 
    'nonsense',
    'splice_acceptor_variant'
]

# 6. The Target Variable
target_col = ['class']

## 5. Check Duplicates

In [27]:
df.duplicated().sum()

np.int64(0)

## 6. Outlier Handling

The boxplots in the visualization notebook revealed outliers in several numeric features. Instead of blanket outlier removal, each column is treated based on what its outliers actually represent:

- **IQR Capping** for `cadd_phred` and `clinical_disease_count`: these have a small, genuine outlier percentage (0.2% and 4.4% respectively). Values beyond 1.5xIQR from Q1/Q3 are capped (not removed), limiting the influence of extreme values while keeping every row in the dataset.
- **`mean_af` is left untouched here.** A standard IQR check flags 18.4% of this column as "outliers", but that's simply the natural shape of allele frequency data (most variants are rare). `mean_af` is also the strongest predictor in the model, so capping these values would destroy the exact signal the model relies on. It gets a log-transform instead, applied later, right before modeling (see the note near the final save below).
- **`loftool`** is left untouched — it has 0 outliers by the IQR test.

In [28]:
def cap_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series.clip(lower, upper)

for col in ['cadd_phred', 'clinical_disease_count']:
    df[col] = cap_outliers_iqr(df[col])

print("Outlier handling complete.")
df[['cadd_phred', 'clinical_disease_count', 'mean_af']].describe()

Outlier handling complete.


,cadd_phred,clinical_disease_count,mean_af
count,65188.000000,65188.000000,65188.000000
mean,15.640675,2.083205,0.014755
std,10.677081,1.534747,0.055338
min,0.001000,0.000000,0.000000
25%,7.304000,1.000000,0.000000
50%,14.090000,2.000000,0.000060
75%,24.000000,3.000000,0.001700
max,49.044000,6.000000,0.484567


## 7. Save Final Cleaned Dataset

This is the canonical cleaned dataset: missing values handled, duplicates checked, irrelevant/leaking columns dropped, features engineered, outliers capped — but categorical columns are still human-readable strings. This is the file used by the EDA notebook (`03_data_visualization.ipynb`) and the Streamlit dashboard (`app.py`).

In [29]:
df.to_csv('/Users/rghdmajd/PycharmProjects/ClinVar_Project/data/processed/cleaned_semi_processed.csv', index=False)
print(f"Saved cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Saved cleaned dataset: 65188 rows, 30 columns


## 8. Data Type Conversion

### Data Type Conversion: Encoding Categorical Features

Before this dataset can be used to train a model, every column must be numeric. Several columns are still stored as text (`str`), and we use three different strategies depending on each column's cardinality and structure:


In [30]:
df.dtypes

chrom                          str
pos                          int64
ref                            str
alt                            str
clnvc                          str
class                        int64
allele                         str
consequence                    str
impact                         str
symbol                         str
feature_type                   str
feature                        str
amino_acids                    str
codons                         str
strand                     float64
loftool                    float64
cadd_phred                 float64
missense_variant             int64
5_prime_utr_variant          int64
intron_variant               int64
3_prime_utr_variant          int64
2kb_upstream_variant         int64
synonymous_variant           int64
500b_downstream_variant      int64
splice_donor_variant         int64
frameshift_variant           int64
nonsense                     int64
splice_acceptor_variant      int64
clinical_disease_cou

In [31]:
# 1. Ordinal Encoding for impact
impact_order = {'MODIFIER': 0, 'LOW': 1, 'MODERATE': 2, 'HIGH': 3}
df['impact'] = df['impact'].map(impact_order)

# 2. One-Hot Encoding
onehot_cols = ['chrom', 'clnvc', 'feature_type']
df = pd.get_dummies(df, columns=onehot_cols, prefix=onehot_cols, drop_first=True, dtype=int)

# 3. Label Encoding
label_cols = ['ref', 'alt', 'allele', 'consequence', 'symbol', 'feature', 'amino_acids', 'codons']
le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Verify
print(df.dtypes)
print(f"Shape after encoding: {df.shape}")

pos                                  int64
ref                                  int64
alt                                  int64
class                                int64
allele                               int64
consequence                          int64
impact                               int64
symbol                               int64
feature                              int64
amino_acids                          int64
codons                               int64
strand                             float64
loftool                            float64
cadd_phred                         float64
missense_variant                     int64
5_prime_utr_variant                  int64
intron_variant                       int64
3_prime_utr_variant                  int64
2kb_upstream_variant                 int64
synonymous_variant                   int64
500b_downstream_variant              int64
splice_donor_variant                 int64
frameshift_variant                   int64
nonsense   

### Log-Transform `mean_af` for Modeling

Unlike `cadd_phred`/`clinical_disease_count`, `mean_af`'s extreme values are the informative rare-variant signal, not noise (see the Outlier Handling section above). Rather than capping it, we apply `log1p` here — right before saving the model-ready dataset — to reduce its skew without discarding information. This transform is only applied to the model-ready file; `cleaned_semi_processed.csv` keeps the raw, human-readable allele frequency.

In [32]:
df['mean_af'] = np.log1p(df['mean_af'])
print("Applied log1p transform to mean_af for modeling.")
df['mean_af'].describe()

Applied log1p transform to mean_af for modeling.


count    65188.000000
mean         0.013381
std          0.048365
min          0.000000
25%          0.000000
50%          0.000060
75%          0.001699
max          0.395123
Name: mean_af, dtype: float64

In [33]:
df.to_csv('/Users/rghdmajd/PycharmProjects/ClinVar_Project/data/processed/clinvar_model_ready.csv', index=False)
print(f"Saved model-ready dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Saved model-ready dataset: 65188 rows, 57 columns


## Key Insights

1. **Missing data was extensive but fixable:** the raw dataset had 46 features, several missing more than 80-99% of values. Dropping columns above a 20% missing threshold and imputing the rest (median for numeric, mode for categorical, 0 for allele frequency) resolved it completely — the final dataset has zero missing values.
2. **Feature engineering unlocked hidden signal:** multi-label text columns like `MC` (variant consequence types) and `CLNDISDB`/`CLNDN` (disease annotations) were unusable as raw strings. Decomposing `MC` into 11 binary flags and converting the disease columns into numeric counts turned unstructured text into features a model can actually learn from.
3. **Leakage and redundancy removed, backed by evidence:** `CLNHGVS` (a near-unique identifier) was dropped to prevent data leakage. The pre-cleaning correlation heatmap showed `CADD_RAW` was almost perfectly correlated with `CADD_PHRED` (r=0.95), and the three allele-frequency columns were highly correlated with each other (r=0.81-0.85) — justifying dropping `CADD_RAW` and consolidating the three AF columns into a single `mean_af`.
4. **No duplicates:** zero duplicate rows were found, so no deduplication was needed.
5. **Outliers handled selectively, not blindly:** `cadd_phred` and `clinical_disease_count` were capped using IQR, but `mean_af`'s extreme values were left untouched (log-transformed only for modeling) since they carry the strongest predictive signal in the whole dataset — capping them would have thrown away exactly the information that matters most for predicting conflicting classifications.
6. **Final result:** from 46 raw, messy columns down to **30 clean, fully-typed columns** (65,188 rows, zero missing values, zero duplicates), plus a separate **57-column fully-encoded** version (`clinvar_model_ready.csv`) ready for modeling.